### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [2]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENROUTER_API_KEY")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

model = init_chat_model("openai:deepseek/deepseek-v4.1-flash", max_tokens=1000)
response = model.invoke("Say hello in one sentence.")
response.content

'Hello!'

In [3]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [4]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content="I'll check that for you." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 297, 'total_tokens': 362, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 22, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 7.241e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 7.241e-05, 'upstream_inference_prompt_cost': 3.861e-05, 'upstream_inference_completions_cost': 3.38e-05}}, 'model_provider': 'openai', 'model_name': 'deepseek/deepseek-v4.1-flash', 'system_fingerprint': None, 'id': 'gen-1789903024-itV75UOxXk9JhWpqHmON', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a0be88-bb87-77c0-a8af-8a74adb55d9a-0' tool_calls=[{'name': 'get_weather', 'args': {'locat

In [5]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is currently **sunny**. Let me know if you'd like the forecast for another location or need anything else!
